# ScholarLens trên Kaggle — crawl, embedding (GPU), Graph, vLLM

Notebook chạy các module **độc lập** của ScholarLens trên GPU Kaggle rồi xuất thư mục `data/` để dùng lại khi demo.

**Chuẩn bị:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.
Đưa mã nguồn lên bằng một trong hai cách: (a) `git clone` repo của bạn, (b) nén thư mục dự án thành Kaggle Dataset tên `scholarlens-code` rồi *Add Input*.

In [ ]:
import os, glob, shutil, subprocess
WORK = "/kaggle/working/ScholarLens"
SRC_DATASET = "/kaggle/input/scholarlens-code"   # cách (b)
GIT_URL = ""                                      # cách (a): điền URL repo nếu có

if GIT_URL:
    !git clone -q {GIT_URL} {WORK}
elif os.path.exists(SRC_DATASET):
    shutil.copytree(SRC_DATASET, WORK, dirs_exist_ok=True)
os.chdir(WORK)
!pip install -q -r requirements.txt
print(os.getcwd(), os.listdir("."))

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
os.environ["SL_EMBED_DEVICE"] = "cuda"
os.environ["SL_EMBED_BATCH"] = "64"      # x4 trên GPU => 256
os.environ["PYTHONIOENCODING"] = "utf-8"

## 1. Thu thập dữ liệu (module `scholarlens.crawler`)
- Theo **keyword**: dùng arXiv API, tối đa vài nghìn bài mỗi truy vấn (nghỉ 3 s/lượt theo quy định arXiv).
- **Độc lập** theo category / khoảng ngày.
- Hoặc nạp số lượng lớn từ Kaggle dataset **Cornell-University/arxiv** (`arxiv-metadata-oai-snapshot.json`).

In [ ]:
TOPICS = [
    ('"retrieval augmented generation"', 600),
    ('"dense retrieval"', 400),
    ('"knowledge graph" "question answering"', 300),
    ('"graph neural network" recommendation', 300),
    ('"large language model" agent', 400),
    ('"text embedding"', 300),
]
for kw, n in TOPICS:
    !python -m scholarlens.crawler keyword '{kw}' --max {n}
!python -m scholarlens.crawler category cs.IR --from 2025-01-01 --max 800
# Bài kinh điển kèm PDF để có quan hệ trích dẫn trong đồ thị
!python -m scholarlens.crawler ids 1706.03762 1810.04805 2005.11401 2004.04906 1908.10084 2002.08909 2007.01282 2112.09118 2212.03533 2309.07597 2310.11511 2401.18059 2104.08663 2007.00808 2208.03299 2305.06983 --pdf
!wc -l data/raw/metadata.jsonl

In [ ]:
# (Tuỳ chọn) nạp từ snapshot Kaggle — bật nếu đã Add Input dataset "Cornell-University/arxiv"
SNAP = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"
if os.path.exists(SNAP):
    !python -m scholarlens.crawler snapshot {SNAP} --categories cs.IR cs.CL --year-from 2024 --limit 20000

## 2. Đăng ký vào SQLite + chunking (module `scholarlens.ingest`)

In [ ]:
!python -m scholarlens.ingest jsonl

## 3. Embedding trên GPU (module `scholarlens.embedding`)

In [ ]:
import time; t=time.time()
!python -m scholarlens.embedding --device cuda
print(f"{time.time()-t:.1f}s")

## 4. Dựng đồ thị tri thức (module `scholarlens.graph`)

In [ ]:
!python -m scholarlens.graph build
!python -m scholarlens.graph related 2005.11401

## 5. Đánh giá truy xuất / dedup / đồ thị

In [ ]:
!python scripts/evaluate.py --n 2000 --models intfloat/multilingual-e5-base BAAI/bge-base-en-v1.5

## 6. Thử Graph RAG ngay trong notebook (chế độ trích đoạn, chưa cần LLM)

In [ ]:
import sys; sys.path.insert(0, WORK)
from scholarlens.services import Services
s = Services.get()
for ev in s.chatbot.respond("Những bài nào liên quan đến dense passage retrieval?", mode="research"):
    if ev["type"] == "token": print(ev["text"])

## 7. Xuất dữ liệu để demo trên máy cá nhân
Tải `scholarlens_data.zip` ở tab Output, giải nén vào thư mục `data/` của dự án.

In [ ]:
shutil.make_archive("/kaggle/working/scholarlens_data", "zip", "data")
!ls -lh /kaggle/working/scholarlens_data.zip

## 8. (Tuỳ chọn) Host LLM bằng vLLM — module `scholarlens.llm_server`
Host bất kỳ model Hugging Face nào vLLM hỗ trợ, có API tương thích OpenAI, mở tunnel **cloudflared** để app trên laptop gọi vào.
- T4 không hỗ trợ bf16 → module tự chọn `float16`; 2×T4 → `tensor-parallel-size=2`.
- Gợi ý: `qwen2.5-7b-awq` (vừa 2×T4), `qwen2.5-3b` (1 GPU). Model gated (Llama) cần Kaggle Secret `HF_TOKEN`.
- Nên chạy **sau** bước embedding (hoặc khởi động lại kernel) để giải phóng VRAM.

In [ ]:
!pip install -q vllm
# from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python -m scholarlens.llm_server --preset qwen2.5-7b-awq --tunnel cloudflared --detach
!cat data/llm_server/llm.env

In [ ]:
# Dùng chính vLLM vừa host cho chatbot trong notebook
from dotenv import dotenv_values
os.environ.update(dotenv_values("data/llm_server/llm.env"))
os.environ["SL_LLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
import importlib, scholarlens.config as cfg; importlib.reload(cfg)
from scholarlens.chat.llm import OpenAICompatLLM
from scholarlens.chat import Chatbot
bot = Chatbot(s.rag, OpenAICompatLLM(os.environ["SL_LLM_MODEL"], os.environ["SL_LLM_BASE_URL"], os.environ["SL_LLM_API_KEY"]))
for ev in bot.respond("RAG khác gì so với REALM? Trả lời có trích dẫn.", mode="research"):
    if ev["type"] == "token": print(ev["text"], end="")

In [ ]:
# Giữ notebook sống để server vLLM tiếp tục phục vụ app trên laptop (dừng: Interrupt)
import time
while True:
    time.sleep(60)